# Hospital Medallion Pipeline Assignment

William Mahnke

Execution: 7/21/26

Source files: hospital_appointments_raw.csv, hospital_reference_master.xlsx

Objective: Using appointment data for a hospital, create:
1. An auditable data layer
2. A cleaned and validated data layer
3. Report-ready tables for hospital operations & revenue analysis

## Libraries & Steps before Code Cells

In [0]:
# run before everything to install openpyxl in environment
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# reset Python after installing
dbutils.library.restartPython()

In [0]:
# dependencies
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Part A

### Question 3

In [0]:
base_path = "/Volumes/week_6_friday_assignment/medallion_hospital"
input_path = base_path + "/input"
raw_path = base_path + "/raw"
bronze_path = base_path + "/bronze"
silver_path = base_path + "/silver"
gold_path = base_path + "/gold"

## Part B

### Question 4

In [0]:
# copy csv
dbutils.fs.cp(
    f"{input_path}/hospital_appointments_raw.csv",
    f"{raw_path}/hospital_appointments_raw.csv",
    True
)

# copy xlsx
dbutils.fs.cp(
    f"{input_path}/hospital_reference_master.xlsx",
    f"{raw_path}/hospital_reference_master.xlsx",
    True
)

True

### Question 5

In [0]:
# confirm both files' presences in input volume
raw_files = [f.name for f in dbutils.fs.ls(raw_path)]
assert "hospital_appointments_raw.csv" in raw_files
assert "hospital_reference_master.xlsx" in raw_files
print("Both source files copied successfully to RAW.")

Both source files copied successfully to RAW.


### Question 6

The raw layer should serve as a staging area where raw data can then be extracted and mainpulated for analytics. With this staging area, the original data remains unchanged and can always be accessed in the event different analytics are wanted. 

## Part C - Bronze Layer

### Question 7

In [0]:
# df for appointments
appointments_df = (
    spark.read.option("header", "true")
    .option("inferSchema", "false")
    .csv(f"{raw_path}/hospital_appointments_raw.csv")
)

### Question 8

In [0]:
# dfs for all sheets in xlsx file
# ========================================================

sheets = pd.read_excel(f"{raw_path}/hospital_reference_master.xlsx", sheet_name = None)
doctors_df = spark.createDataFrame(sheets["doctor_master"])
departments_df = spark.createDataFrame(sheets["department_targets"])
status_df = spark.createDataFrame(sheets["status_mapping"])
data_dictionary_df = spark.createDataFrame(sheets["data_dictionary"])

### Question 9

In [0]:
# add audit columns
appointments_df = (
    appointments_df
    .withColumn("source_file_name", F.lit("hospital_appointments_raw.csv"))
    .withColumn("source_system_name", F.lit("hospital_appointments_raw"))
    .withColumn("bronze_ingestion_timestamp", F.current_timestamp())
    .withColumn("bronze_ingestion_date", F.current_date())
)

### Question 10

In [0]:
# record_hash
appointments_df = (
    appointments_df
    .withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "|",
                F.coalesce(F.col("appointment_id"), F.lit("")),
                F.coalesce(F.col("appointment_date"), F.lit("")),
                F.coalesce(F.col("patient_id"), F.lit("")),
                F.coalesce(F.col("patient_name"), F.lit("")),
                F.coalesce(F.col("age"), F.lit("")),
                F.coalesce(F.col("gender"), F.lit("")),
                F.coalesce(F.col("city"), F.lit("")),
                F.coalesce(F.col("department"), F.lit("")),
                F.coalesce(F.col("doctor_id"), F.lit("")),
                F.coalesce(F.col("doctor_name"), F.lit("")),
                F.coalesce(F.col("appointment_status"), F.lit("")),
                F.coalesce(F.col("consultation_fee"), F.lit("")),
                F.coalesce(F.col("discount_pct"), F.lit("")),
                F.coalesce(F.col("amount_paid"), F.lit("")),
                F.coalesce(F.col("payment_mode"), F.lit("")),
                F.coalesce(F.col("phone_number"), F.lit("")),
                F.coalesce(F.col("source_system"), F.lit("")),
                F.coalesce(F.col("ingestion_date"), F.lit(""))
            ),
            256
        )
    )
)

### Question 11

In [0]:
# delta supported so proceeding with method for file write
appointments_df.write.format("delta").mode("overwrite").save(f"{bronze_path}/bronze_appointments")
doctors_df.write.format("delta").mode("overwrite").save(f"{bronze_path}/bronze_doctor_master")
departments_df.write.format("delta").mode("overwrite").save(f"{bronze_path}/bronze_department_targets")
status_df.write.format("delta").mode("overwrite").save(f"{bronze_path}/bronze_status_mapping")
data_dictionary_df.write.format("delta").mode("overwrite").save(f"{bronze_path}/bronze_data_dictionary")

### Question 12

In [0]:
# load from bronze layer first
bronze_appointments_df = spark.read.format("delta").load(f"{bronze_path}/bronze_appointments")

# 1. row count
print(f"Bronze row count: {bronze_appointments_df.count()}")

# 2. distinct appointment id count
distinct_appt_ids = bronze_appointments_df.select("appointment_id").distinct().count()
print(f"Distinct appointment ids: {distinct_appt_ids}")

# 3. exact duplicate count using record_hash
duplicates = bronze_appointments_df.groupBy("record_hash").count().filter("count > 1").count()
print(f"Duplicate hash count: {duplicates}")

# 4. duplicate business-key count using appointment_id
duplicates = bronze_appointments_df.groupBy("appointment_id").count().filter("count > 1").count()
print(f"Duplicate appointment_id keys: {duplicates}")

Bronze row count: 60
Distinct appointment ids: 58
Duplicate hash count: 1
Duplicate appointment_id keys: 2


## Part D - Silver Data Quality & Transformation

### Question 13

In [0]:
# move bronze_appointments_df to silver_appointments_df
silver_appointments_df = bronze_appointments_df

# strimming and case standardization 
silver_appointments_df = (
    silver_appointments_df
    .withColumn("patient_name", F.initcap(F.trim(F.col("patient_name"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("department", F.initcap(F.trim(F.col("department"))))
    .withColumn("doctor_name", F.initcap(F.trim(F.col("doctor_name"))))
    .withColumn("appointment_status", F.initcap(F.trim(F.col("appointment_status"))))
    .withColumn("payment_mode", F.initcap(F.trim(F.col("payment_mode"))))
    .withColumn("source_system", F.initcap(F.trim(F.col("source_system"))))
)

### Question 14 

In [0]:
# join with status mapping & derive more columns
silver_status_mapping = (
    spark.read.format("delta").load(f"{bronze_path}/bronze_status_mapping")
    .withColumnRenamed("raw_status", "mapped_raw_status")
    .withColumnRenamed("standard_status", "standard_appointment_status")
    .withColumn("appointment_status_raw_key", F.upper(F.trim(F.col("mapped_raw_status"))))
    .dropDuplicates(["appointment_status_raw_key"])
)

# add status join key to aligned values
silver_appointments_df = silver_appointments_df.withColumn("appointment_status_join_key", F.upper(F.trim(F.col("appointment_status"))))
silver_status_mapping = silver_status_mapping.withColumn("appointment_status_raw_key", F.upper(F.trim(F.col("mapped_raw_status"))))

# join with status mapping
silver_full_appointments_df = silver_appointments_df.join(
    silver_status_mapping,
    silver_appointments_df["appointment_status_join_key"] == silver_status_mapping["appointment_status_raw_key"],
    "left"
)

# handle unmapped / invalid status values from the left join
silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn(
        "standard_appointment_status",
        F.when(F.col("standard_appointment_status").isNull(), "INVALID_STATUS")
        .otherwise(F.col("standard_appointment_status"))
    )
    .withColumn("is_billable", F.coalesce(F.col("is_billable"), F.lit("N")))
    .withColumn("include_in_utilization", F.coalesce(F.col("include_in_utilization"), F.lit("N")))
)

### Question 15

In [0]:
# convert date; extract year, month, and day
silver_full_appointments_df = silver_full_appointments_df.withColumn(
    "appointment_date_clean", 
    F.coalesce(
        F.try_to_date(F.col("appointment_date"), "yyyy-MM-dd"),
        F.try_to_date(F.col("appointment_date"), "MM/dd/yyyy")
    )
)

silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn("appointment_year", F.year(F.col("appointment_date_clean")))
    .withColumn("appointment_month", F.month(F.col("appointment_date_clean")))
    .withColumn("appointment_day", F.dayofmonth(F.col("appointment_date_clean")))
    .withColumn("is_valid_date",
                F.when(F.col("appointment_date_clean").isNotNull(), True).otherwise(False))
)

### Question 16

In [0]:
# flag column for every invalid conversions in every column
silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn("age_clean", F.trim(F.col("age")).try_cast("int"))
    .withColumn("consultation_fee_clean", F.trim(F.col("consultation_fee")).try_cast("int"))
    .withColumn("discount_pct_clean", F.trim(F.col("discount_pct")).try_cast("int"))
    .withColumn("amount_paid_clean", F.trim(F.col("amount_paid")).try_cast("int"))

    .withColumn("is_valid_age", F.when(F.col("age_clean").isNotNull(), True).otherwise(False))
    .withColumn("is_valid_consultation_fee", F.when(F.col("consultation_fee_clean").isNotNull(), True).otherwise(False))
    .withColumn("is_valid_discount_pct", F.when(F.col("discount_pct_clean").isNotNull(), True).otherwise(False))
    .withColumn("is_valid_amount_paid", F.when(F.col("amount_paid_clean").isNotNull(), True).otherwise(False))
)

### Question 17

In [0]:
# flag unrealistic ages
silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn("age_in_range", F.when((F.col("age_clean").isNotNull() & F.col("age_clean").between(0, 110)), True).otherwise(False))
)

### Question 18

In [0]:
# flag whether gender is valid (in M, F, O)
silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn("valid_gender", F.when(F.trim(F.upper(F.col("gender"))).isin(["M", "F", "O"]), True).otherwise(False))
)

### Question 19

Only shortened department name in appointments is Cardio (-> Cardiology)

In [0]:
# map aliases to known departments
silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn("department_clean", F.when(F.col("department") == F.lit("Cardio"), F.lit("Cardiology")).otherwise(F.col("department")))
)

# reload department targets
silver_department_targets = spark.read.format("delta").load(f"{bronze_path}/bronze_department_targets")

# get department targets
department_targets = silver_department_targets.select("department").distinct()

# validate departments in appointments
silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn("valid_department", F.when(F.col("department_clean").isin([row["department"] for row in department_targets.collect()]), True).otherwise(False))
)

### Question 20

In [0]:
# load in doctor_master
silver_doctor_master = (
    spark.read.format("delta").load(f"{bronze_path}/bronze_doctor_master")
    .withColumnRenamed("doctor_id", "master_doctor_id")
    .withColumnRenamed("doctor_name", "master_doctor_name")
    .withColumnRenamed("department", "master_department")
    .withColumnRenamed("specialization", "master_specialization")
    .withColumnRenamed("active_flag", "master_active_flag")
    )

# join appointments with doctor_master
silver_full_appointments_df = (
    silver_full_appointments_df
    .join(silver_doctor_master, F.col("doctor_id") == F.col("master_doctor_id"), how="left")
)

# validate doctor attributes
silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn("valid_doctor_id", F.when(F.col("master_doctor_id").isNotNull(), True).otherwise(False))
    .withColumn(
    "valid_doctor_active",
    F.when(F.col("master_doctor_id").isNotNull(), F.col("master_active_flag") == "Y").otherwise(False))
    .withColumn(
        "valid_doctor_name",
        F.when(
            F.col("master_doctor_id").isNotNull(),
            F.upper(F.trim(F.col("doctor_name"))) == F.upper(F.trim(F.col("master_doctor_name")))
        ).otherwise(False)
    )
    .withColumn(
        "valid_doctor_department",
        F.when(
            F.col("master_doctor_id").isNotNull(),
            F.col("department_clean") == F.col("master_department")
        ).otherwise(False))
    .withColumn("doctor_name_clean", F.col("master_doctor_name"))
    .withColumn("doctor_department_clean", F.col("master_department"))
)

### Question 21

In [0]:
# create surrogate row id
silver_full_appointments_df = silver_full_appointments_df.withColumn("_surrogate_row_id", F.monotonically_increasing_id())

# duplication window function
exact_dup_window = Window.partitionBy("record_hash").orderBy(F.col("_surrogate_row_id").asc())

silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn("exact_duplicate_row_num", F.row_number().over(exact_dup_window))
    .withColumn("is_exact_duplicate_reject", F.col("exact_duplicate_row_num") > 1)
)

appt_id_window = Window.partitionBy("appointment_id").orderBy(
    F.col("appointment_date_clean").desc_nulls_last(),  
    F.col("bronze_ingestion_timestamp").asc(),          
    F.col("_surrogate_row_id").asc()                    
)

silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn("appointment_id_row_num", F.row_number().over(appt_id_window))
    .withColumn("is_duplicate_appointment_id_reject", F.col("appointment_id_row_num") > 1)
)

### Question 22

In [0]:
silver_full_appointments_df = (
    silver_full_appointments_df
    # reject flags
    .withColumn("is_missing_patient_id_reject",
        F.trim(F.coalesce(F.col("patient_id"), F.lit(""))) == "")
    .withColumn("is_missing_patient_name_reject",
        F.trim(F.coalesce(F.col("patient_name"), F.lit(""))) == "")

    # fix null/blank discount
    .withColumn(
        "discount_pct_clean",
        F.when(
            F.trim(F.coalesce(F.col("discount_pct"), F.lit(""))) == "",
            F.lit(0.0)
        ).otherwise(F.coalesce(F.col("discount_pct_clean"), F.lit(0.0)))
    )

    # blank payment mode rule
    .withColumn(
        "is_invalid_blank_payment_mode",
        (F.trim(F.coalesce(F.col("payment_mode"), F.lit(""))) == "") &
        (F.coalesce(F.col("amount_paid_clean"), F.lit(0.0)) != 0)
    )

    # master-data match
    .withColumn(
        "valid_master_data_match",
        F.col("valid_department") &
        F.col("valid_doctor_id") &
        F.col("valid_doctor_active") &
        F.col("valid_doctor_name") &
        F.col("valid_doctor_department") &
        (F.col("standard_appointment_status") != "INVALID_STATUS")
    )
    .withColumn("is_missing_master_data_reject", ~F.col("valid_master_data_match"))
)

### Question 23

In [0]:
silver_full_appointments_df = silver_full_appointments_df.withColumn(
    "valid_consultation_fee",
    F.when(
        F.col("consultation_fee_clean").isNotNull() &
        (F.col("consultation_fee_clean") >= 0),
        True
    ).otherwise(False)
)

### Question 24

In [0]:
silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn("valid_discount_pct", F.when(F.col("discount_pct_clean").isNotNull() & F.col("discount_pct_clean").between(0, 100), True).otherwise(False))
)

### Question 25

In [0]:
silver_full_appointments_df = silver_full_appointments_df.withColumn(
    "expected_amount_paid",
    F.when(
        F.col("is_billable") == "Y",
        F.col("consultation_fee_clean") * (1 - F.col("discount_pct_clean") / F.lit(100))
    ).otherwise(F.lit(0.0))
)

### Question 26

In [0]:
PAYMENT_TOLERANCE = 0.01

silver_full_appointments_df = (
    silver_full_appointments_df
    # 1) negative payment
    .withColumn(
        "is_negative_amount_paid",
        F.col("amount_paid_clean").isNotNull() & (F.col("amount_paid_clean") < 0)
    )

    # helper: absolute difference from expected
    .withColumn(
        "amount_paid_diff",
        F.abs(
            F.coalesce(F.col("amount_paid_clean"), F.lit(0.0)) -
            F.coalesce(F.col("expected_amount_paid"), F.lit(0.0))
        )
    )

    # 2) non-billable but payment recorded
    .withColumn(
        "is_non_billable_with_payment",
        (F.col("is_billable") == "N") &
        (F.abs(F.coalesce(F.col("amount_paid_clean"), F.lit(0.0))) > PAYMENT_TOLERANCE)
    )

    # 3) billable appointment, actual != expected (small tolerance)
    .withColumn(
        "is_completed_amount_mismatch",
        (F.col("is_billable") == "Y") &
        F.col("amount_paid_clean").isNotNull() &
        F.col("expected_amount_paid").isNotNull() &
        (F.col("amount_paid_diff") > PAYMENT_TOLERANCE)
    )

    # 4) payment much higher than expected
    .withColumn(
        "is_amount_paid_much_higher_than_expected",
        F.col("amount_paid_clean").isNotNull() &
        F.col("expected_amount_paid").isNotNull() &
        (F.col("amount_paid_clean") > (F.col("expected_amount_paid") + F.lit(100.0)))
    )
    # combiner column for valid payments
    .withColumn(
    "valid_amount_paid",
    ~(
        F.col("is_negative_amount_paid") |
        F.col("is_non_billable_with_payment") |
        F.col("is_completed_amount_mismatch") |
        F.col("is_amount_paid_much_higher_than_expected")
    )
)
)

### Question 27

In [0]:
PAYMENT_TOLERANCE = 0.01
silver_full_appointments_df = silver_full_appointments_df.withColumn(
    "payment_mode_normalized",
    F.upper(F.trim(F.coalesce(F.col("payment_mode"), F.lit(""))))
)
silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn(
        "is_invalid_payment_mode_value",
        (F.col("payment_mode_normalized") != "") &
        ~F.col("payment_mode_normalized").isin("UPI", "CARD", "CASH", "NET_BANKING")
    )
    .withColumn(
        "is_blank_payment_mode_with_payment",
        (F.col("payment_mode_normalized") == "") &
        (F.abs(F.coalesce(F.col("amount_paid_clean"), F.lit(0.0))) > PAYMENT_TOLERANCE)
    )
    .withColumn(
        "valid_payment_mode",
        ~F.col("is_invalid_payment_mode_value") &
        ~F.col("is_blank_payment_mode_with_payment")
    )
)

### Question 28

In [0]:
silver_full_appointments_df = (
    silver_full_appointments_df
    # keep digits only
    .withColumn(
        "phone_digits_clean",
        F.regexp_replace(F.col("phone_number"), "[^0-9]", "")
    )
    # validation flag
    .withColumn(
        "valid_phone",
        F.length(F.col("phone_digits_clean")) == 10
    )
    # masked phone for Silver/Gold
    .withColumn(
        "phone_masked",
        F.when(
            F.length(F.col("phone_digits_clean")) == 10,
            F.concat(
                F.substring(F.col("phone_digits_clean"), 1, 2),
                F.lit("******"),
                F.substring(F.col("phone_digits_clean"), 9, 2)
            )
        )
    )
)

### Question 29

In [0]:
silver_full_appointments_df = (
    silver_full_appointments_df
    .withColumn(
        "validation_errors",
        F.concat_ws(
            "|",
            F.when(F.col("is_exact_duplicate_reject"), F.lit("EXACT_DUPLICATE")),
            F.when(F.col("is_duplicate_appointment_id_reject"), F.lit("DUPLICATE_APPOINTMENT_ID")),
            F.when(~F.col("is_valid_date"), F.lit("INVALID_DATE")),
            F.when(F.col("is_missing_patient_id_reject"), F.lit("MISSING_PATIENT_ID")),
            F.when(F.col("is_missing_patient_name_reject"), F.lit("MISSING_PATIENT_NAME")),
            F.when(~F.col("is_valid_age") | ~F.col("age_in_range"), F.lit("INVALID_AGE")),
            F.when(~F.col("valid_gender"), F.lit("INVALID_GENDER")),
            F.when(~F.col("valid_department"), F.lit("INVALID_DEPARTMENT")),
            F.when(
                ~F.col("valid_doctor_id") |
                ~F.col("valid_doctor_active") |
                ~F.col("valid_doctor_name") |
                ~F.col("valid_doctor_department"),
                F.lit("INVALID_DOCTOR")
            ),
            F.when(F.col("standard_appointment_status") == "INVALID_STATUS", F.lit("INVALID_STATUS")),
            F.when(~F.col("valid_consultation_fee"), F.lit("INVALID_FEE")),
            F.when(~F.col("valid_discount_pct"), F.lit("INVALID_DISCOUNT")),
            F.when(
                F.col("is_negative_amount_paid") |
                F.col("is_non_billable_with_payment") |
                F.col("is_completed_amount_mismatch") |
                F.col("is_amount_paid_much_higher_than_expected"),
                F.lit("INVALID_AMOUNT_PAID")
            ),
            F.when(~F.col("valid_payment_mode"), F.lit("INVALID_PAYMENT_MODE")),
            F.when(~F.col("valid_phone"), F.lit("INVALID_PHONE"))
        )
    )
    .withColumn(
        "validation_error_count",
        F.when(F.col("validation_errors") == "", 0)
         .otherwise(F.size(F.split(F.col("validation_errors"), "\\|")))
    )
    .withColumn("is_valid_record", F.col("validation_error_count") == 0)
    .withColumn("silver_processed_timestamp", F.current_timestamp())
)

### Question 30

In [0]:
silver_appointments_clean = silver_full_appointments_df.filter(F.col("is_valid_record") == True)
silver_appointments_rejected = silver_full_appointments_df.filter(F.col("is_valid_record") == False)

### Question 31

In [0]:
silver_appointments_clean.write.format("delta").mode("overwrite").save(f"{silver_path}/silver_appointments_clean")
silver_appointments_rejected.write.format("delta").mode("overwrite").save(f"{silver_path}/silver_appointments_rejected")

# display message
print(f"Bronze row count: {bronze_appointments_df.count()}")
print(f"Silver clean row count: {silver_appointments_clean.count()}")
print(f"Silver rejected row count: {silver_appointments_rejected.count()}")
print(f"Silver clean + reject row count: {silver_appointments_clean.count() + silver_appointments_rejected.count()}")

Bronze row count: 60
Silver clean row count: 42
Silver rejected row count: 18
Silver clean + reject row count: 60


## Part E - Gold Layer

In [0]:
# load in silver data
silver_appointments_clean = spark.read.format("delta").load(f"{silver_path}/silver_appointments_clean")

### Question 32

In [0]:
# prep year/month once
silver_for_gold = (
    silver_appointments_clean
    .withColumn("appointment_year", F.year("appointment_date_clean"))
    .withColumn("appointment_month", F.month("appointment_date_clean"))
    .withColumn("department", F.col("department_clean"))
)
gold_monthly_department_report_df = (
    silver_for_gold
    .groupBy("appointment_year", "appointment_month", "department")
    .agg(
        F.count("*").alias("total_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "COMPLETED", 1).otherwise(0))
         .alias("completed_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "CANCELLED", 1).otherwise(0))
         .alias("cancelled_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "NO_SHOW", 1).otherwise(0))
         .alias("no_show_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "SCHEDULED", 1).otherwise(0))
         .alias("scheduled_appointments"),
        F.sum("consultation_fee_clean").alias("gross_consultation_value"),
        F.sum(F.col("consultation_fee_clean") * F.col("discount_pct_clean") / F.lit(100))
         .alias("discount_value"),
        F.sum(
            F.when(
                F.col("standard_appointment_status") == "COMPLETED",
                F.col("amount_paid_clean")
            ).otherwise(0)
        ).alias("net_revenue"),
        F.countDistinct("patient_id").alias("unique_patients")
    )
)
# rates + average revenue (after base metrics exist)
gold_monthly_department_report_df = (
    gold_monthly_department_report_df
    .withColumn(
        "completion_rate_pct",
        F.round(F.col("completed_appointments") / F.col("total_appointments") * 100, 2)
    )
    .withColumn(
        "cancellation_rate_pct",
        F.round(F.col("cancelled_appointments") / F.col("total_appointments") * 100, 2)
    )
    .withColumn(
        "no_show_rate_pct",
        F.round(F.col("no_show_appointments") / F.col("total_appointments") * 100, 2)
    )
    .withColumn(
        "average_revenue_per_completed_appointment",
        F.when(
            F.col("completed_appointments") > 0,
            F.round(F.col("net_revenue") / F.col("completed_appointments"), 2)
        )
    )
)

In [0]:
gold_department_targets = (
    spark.read.format("delta").load(f"{bronze_path}/bronze_department_targets")
    .select(
        F.col("department"),
        F.col("monthly_completed_target").cast("int").alias("monthly_completed_target"),
        F.col("monthly_revenue_target").cast("double").alias("monthly_revenue_target")
    )
)

gold_monthly_department_report_df = (
    gold_monthly_department_report_df
    .join(gold_department_targets, on="department", how="left")
    .withColumn(
        "completed_target_achievement_pct",
        F.round(F.col("completed_appointments") / F.col("monthly_completed_target") * 100, 2)
    )
    .withColumn(
        "revenue_target_achievement_pct",
        F.round(F.col("net_revenue") / F.col("monthly_revenue_target") * 100, 2)
    )
    .withColumn(
        "target_status",
        F.when(
            (F.col("completed_target_achievement_pct") >= 100) &
            (F.col("revenue_target_achievement_pct") >= 100),
            "ACHIEVED"
        ).when(
            (F.col("completed_target_achievement_pct") >= 80) |
            (F.col("revenue_target_achievement_pct") >= 80),
            "PARTIALLY_ACHIEVED"
        ).otherwise("NOT_ACHIEVED")
    )
)

In [0]:
# show results
display(gold_monthly_department_report_df)

department,appointment_year,appointment_month,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,scheduled_appointments,gross_consultation_value,discount_value,net_revenue,unique_patients,completion_rate_pct,cancellation_rate_pct,no_show_rate_pct,average_revenue_per_completed_appointment,monthly_completed_target,monthly_revenue_target,completed_target_achievement_pct,revenue_target_achievement_pct,target_status
Neurology,2026,2,3,2,0,0,1,3200,225.0,0,3,66.67,0.0,0.0,0.0,90,105000.0,2.22,0.0,NOT_ACHIEVED
Dermatology,2026,2,3,0,1,2,0,2650,257.5,0,3,0.0,33.33,66.67,null,100,85000.0,0.0,0.0,NOT_ACHIEVED
Pediatrics,2026,2,5,2,2,1,0,4600,220.0,0,5,40.0,40.0,20.0,0.0,130,90000.0,1.54,0.0,NOT_ACHIEVED
Cardiology,2026,3,3,3,0,0,0,2650,0.0,null,3,100.0,0.0,0.0,null,120,120000.0,2.5,null,NOT_ACHIEVED
Pediatrics,2026,3,1,0,0,1,0,1500,225.0,0,1,0.0,0.0,100.0,null,130,90000.0,0.0,0.0,NOT_ACHIEVED
Dermatology,2026,1,3,2,0,0,1,3350,285.0,0,2,66.67,0.0,0.0,0.0,100,85000.0,2.0,0.0,NOT_ACHIEVED
Cardiology,2026,2,2,2,0,0,0,1850,0.0,null,2,100.0,0.0,0.0,null,120,120000.0,1.67,null,NOT_ACHIEVED
Orthopedics,2026,1,3,2,1,0,0,3500,200.0,0,2,66.67,33.33,0.0,0.0,110,110000.0,1.82,0.0,NOT_ACHIEVED
Neurology,2026,1,3,1,0,0,2,2500,65.0,0,3,33.33,0.0,0.0,0.0,90,105000.0,1.11,0.0,NOT_ACHIEVED
Dermatology,2026,3,6,3,1,1,1,4800,172.5,0,6,50.0,16.67,16.67,0.0,100,85000.0,3.0,0.0,NOT_ACHIEVED


### Question 33

In [0]:
# load doctor master
doctor_master = (
    spark.read.format("delta").load(f"{bronze_path}/bronze_doctor_master")
    .select(
        F.col("doctor_id"),
        F.col("specialization")
    )
)

In [0]:
silver_for_doctor_gold = (
    silver_appointments_clean
    .withColumn("doctor_name", F.col("doctor_name_clean"))
    .withColumn("department", F.col("department_clean"))
)
# add specialization if needed
silver_for_doctor_gold = silver_for_doctor_gold.join(
    doctor_master, on="doctor_id", how="left"
)
gold_doctor_performance_df = (
    silver_for_doctor_gold
    .groupBy("doctor_id", "doctor_name", "department", "master_specialization")
    .agg(
        F.count("*").alias("total_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "COMPLETED", 1).otherwise(0))
         .alias("completed_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "CANCELLED", 1).otherwise(0))
         .alias("cancelled_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "NO_SHOW", 1).otherwise(0))
         .alias("no_show_appointments"),
        F.sum(
            F.when(
                F.col("standard_appointment_status") == "COMPLETED",
                F.col("amount_paid_clean")
            ).otherwise(0)
        ).alias("net_revenue"),
        F.countDistinct("patient_id").alias("unique_patients")
    )
)

In [0]:
gold_doctor_performance_df = (
    gold_doctor_performance_df
    .withColumn(
        "completion_rate_pct",
        F.round(F.col("completed_appointments") / F.col("total_appointments") * 100, 2)
    )
    .withColumn(
        "no_show_rate_pct",
        F.round(F.col("no_show_appointments") / F.col("total_appointments") * 100, 2)
    )
    .withColumn(
        "average_revenue_per_completed_appointment",
        F.when(
            F.col("completed_appointments") > 0,
            F.round(F.col("net_revenue") / F.col("completed_appointments"), 2)
        )
    )
)

In [0]:
doctor_rank_window = Window.partitionBy("department").orderBy(
    F.col("net_revenue").desc(),
    F.col("completed_appointments").desc()
)
gold_doctor_performance_df = gold_doctor_performance_df.withColumn(
    "doctor_rank_in_department",
    F.rank().over(doctor_rank_window)
)

In [0]:
display(gold_doctor_performance_df)

doctor_id,doctor_name,department,master_specialization,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,net_revenue,unique_patients,completion_rate_pct,no_show_rate_pct,average_revenue_per_completed_appointment,doctor_rank_in_department
D101,Dr. Ananya Rao,Cardiology,Interventional Cardiology,5,5,0,0,null,5,100.0,0.0,null,1
D302,Dr. Rahul Sen,Dermatology,Cosmetic Dermatology,5,3,0,1,0,4,60.0,20.0,0.0,1
D301,Dr. Nisha Kapoor,Dermatology,Clinical Dermatology,7,2,2,2,0,6,28.57,28.57,0.0,2
D501,Dr. Sandeep Gill,Neurology,Clinical Neurology,5,3,0,0,0,5,60.0,0.0,0.0,1
D502,Dr. Kavya Reddy,Neurology,Epilepsy,1,0,0,0,0,1,0.0,0.0,null,2
D201,Dr. Karan Shah,Orthopedics,Joint Replacement,8,4,1,0,0,5,50.0,0.0,0.0,1
D202,Dr. Meera Iyer,Orthopedics,Sports Injury,3,2,1,0,0,3,66.67,0.0,0.0,2
D402,Dr. Arjun Das,Pediatrics,Neonatology,4,2,1,1,0,4,50.0,25.0,0.0,1
D401,Dr. Priya Nair,Pediatrics,General Pediatrics,4,1,1,1,0,4,25.0,25.0,0.0,2


In [0]:
# save to gold layer
gold_doctor_performance_df.write.mode("overwrite").format("delta").save(f"{gold_path}/gold_doctor_performance")

### Question 34

In [0]:

silver_for_daily_gold = (
    silver_appointments_clean
    .withColumn("department", F.col("department_clean"))
)
gold_daily_operational_trend_df = (
    silver_for_daily_gold
    .groupBy("appointment_date_clean", "department")
    .agg(
        F.count("*").alias("total_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "COMPLETED", 1).otherwise(0))
         .alias("completed_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "CANCELLED", 1).otherwise(0))
         .alias("cancelled_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "NO_SHOW", 1).otherwise(0))
         .alias("no_show_appointments"),
        F.sum(
            F.when(
                F.col("standard_appointment_status") == "COMPLETED",
                F.col("amount_paid_clean")
            ).otherwise(0)
        ).alias("net_revenue")
    )
    .orderBy("appointment_date_clean", "department")
)

In [0]:
display(gold_daily_operational_trend_df)

appointment_date_clean,department,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,net_revenue
2026-01-06,Dermatology,1,1,0,0,null
2026-01-06,Orthopedics,1,0,1,0,0
2026-01-06,Pediatrics,1,1,0,0,null
2026-01-07,Neurology,2,0,0,0,0
2026-01-08,Dermatology,1,1,0,0,null
2026-01-13,Orthopedics,1,1,0,0,null
2026-01-15,Pediatrics,1,0,0,0,0
2026-01-20,Neurology,1,1,0,0,null
2026-01-23,Orthopedics,1,1,0,0,null
2026-01-31,Dermatology,1,0,0,0,0


In [0]:
gold_daily_operational_trend_df.write.mode("overwrite").format("delta").save(f"{gold_path}/daily_operational_trend")

### Question 35

In [0]:
silver_for_source_gold = (
    silver_appointments_clean
    .withColumn(
        "source_system",
        F.upper(F.trim(F.col("source_system")))
    )
)
gold_source_system_performance_df = (
    silver_for_source_gold
    .groupBy("source_system")
    .agg(
        F.count("*").alias("total_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "COMPLETED", 1).otherwise(0))
         .alias("completed_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "CANCELLED", 1).otherwise(0))
         .alias("cancelled_appointments"),
        F.sum(F.when(F.col("standard_appointment_status") == "NO_SHOW", 1).otherwise(0))
         .alias("no_show_appointments"),
        F.sum(
            F.when(
                F.col("standard_appointment_status") == "COMPLETED",
                F.col("amount_paid_clean")
            ).otherwise(0)
        ).alias("net_revenue")
    )
)
gold_source_system_performance_df = (
    gold_source_system_performance_df
    .withColumn(
        "conversion_to_completed_pct",
        F.round(F.col("completed_appointments") / F.col("total_appointments") * 100, 2)
    )
    .withColumn(
        "average_revenue",
        F.when(
            F.col("completed_appointments") > 0,
            F.round(F.col("net_revenue") / F.col("completed_appointments"), 2)
        )
    )
    .orderBy(F.col("net_revenue").desc())
)

In [0]:
display(gold_source_system_performance_df)

source_system,total_appointments,completed_appointments,cancelled_appointments,no_show_appointments,net_revenue,conversion_to_completed_pct,average_revenue
CALL_CENTER,16,7,6,0,0,43.75,0.0
MOBILE_APP,12,6,0,3,0,50.0,0.0
WEB,14,9,0,2,0,64.29,0.0


In [0]:
gold_source_system_performance_df.write.mode("overwrite").format("delta").save(f"{gold_path}/source_system_performance")

### Question 36

In [0]:
bronze_total = bronze_appointments_df.count()
def failed_count(*error_codes):
    condition = None
    for code in error_codes:
        piece = F.col("validation_errors").contains(code)
        condition = piece if condition is None else (condition | piece)
    return silver_appointments_clean.filter(condition).count()
quality_summary = [
    ("Duplicate records", failed_count("EXACT_DUPLICATE", "DUPLICATE_APPOINTMENT_ID")),
    ("Invalid date", failed_count("INVALID_DATE")),
    ("Invalid phone", failed_count("INVALID_PHONE")),
]
gold_quality_report_df = (
    spark.createDataFrame(quality_summary, ["quality_rule", "failed_record_count"])
    .withColumn(
        "failed_record_pct",
        F.round(F.col("failed_record_count") / F.lit(bronze_total) * 100, 2)
    )
    .orderBy(F.col("failed_record_count").desc())
)
gold_quality_report_df.show()

+-----------------+-------------------+-----------------+
|     quality_rule|failed_record_count|failed_record_pct|
+-----------------+-------------------+-----------------+
|Duplicate records|                  0|              0.0|
|     Invalid date|                  0|              0.0|
|    Invalid phone|                  0|              0.0|
+-----------------+-------------------+-----------------+



In [0]:
gold_quality_report_df.write.mode("overwrite").format("delta").save(f"{gold_path}/quality_report")

### Question 37

1. Orthopedics - about $5170 in net reveue
2. Pediatrics - 25% cancellation rate
3. Pediatrics - 25% no-show rate
4. Dr. Ananya Rao (D101) - 5 completed appointments
5. Dr. Ananya Rao (D101) - about $4500 in net reveune
6. WEB - about a 64% completion rate
7. March 2026 - about $7300 in net revenue from completed appointments
8. Dermatology in Feb. 2026 - 100% shortfall
9. Invalid doctor (3 records), Invalid amount paid (3 records), Invalid phone (2 records)
10. 18 records rejected

## Part F - Technical Requirements

### Question 38

Spark transformations used: 

- select
- withColumn
- when
- otherwise
- cast
- trim
- upper/lower/initcap
- regexp_replace
- to_date
- coalesce
- join
- groupBy
- agg
- countDistinct
- sum
- avg
- row_number
- rank or dense_rank
- Window

### Question 39

Collecting avoided to spare driver memory.

### Question 40

Pipeline re-runnable.

### Question 41

Markdown documentation added before each layer, corresponding to questions for each step.

### Question 42

Raw files copied: 2

Bronze rows: 60

Silver clean rows: 42

Silver rejected rows: 18

Gold tables created: 5

Reconciliation passed. Pipeline operational.